# 02b — Competitor Feature Extraction

**Source data:** `data/raw/export.geojson` — 1,227 restaurant points scraped from OpenStreetMap via Overpass API for Delhi region.

**What this notebook does:** For every 500m x 500m grid cell, computes:
- How many competitors are inside the cell
- Distance to the nearest competitor
- Competitor density in a 1km radius
- Whether any delivery-enabled restaurant exists nearby
- Cuisine diversity score
- Chain vs independent restaurant ratio

**Input :** `data/raw/export.geojson` + `data/processed/grid_with_population.pkl`  
**Output:** `data/processed/grid_with_competitors.pkl` — ready for `02_features.ipynb`

**Run order:** `01_grid_build` → `02a_worldpop` → **`02b_competitor`** → `02_features` → `03_train` → `04_bip`

## 0. Install dependencies (run once)

In [ ]:
# Uncomment if needed
# !pip install geopandas shapely scipy numpy pandas matplotlib --quiet

## 1. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from shapely.geometry  import Point
from scipy.spatial     import cKDTree

warnings.filterwarnings('ignore')
print('All imports OK.')

## 2. Configuration

In [ ]:
# ── Bounding box — must match 01_grid_build.ipynb exactly ─────────────
MIN_LON = 76.8
MIN_LAT = 28.4
MAX_LON = 77.4
MAX_LAT = 28.9

# ── Paths ──────────────────────────────────────────────────────────────
COMPETITOR_GEOJSON = 'data/raw/export.geojson'
GRID_INPUT         = 'data/processed/grid_with_population.pkl'  # from 02a
GRID_FALLBACK      = 'data/processed/grid.geojson'              # if 02a not run yet
OUT_PKL            = 'data/processed/grid_with_competitors.pkl'

# ── Competitor feature parameters ─────────────────────────────────────
# Radius for density count — how far we look beyond the cell boundary
DENSITY_RADIUS_DEG = 0.009   # ~1 km in degrees at Delhi latitude

os.makedirs('data/raw',       exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('Config ready.')
print(f'  Competitor source : {COMPETITOR_GEOJSON}')
print(f'  Delhi bbox        : ({MIN_LON}, {MIN_LAT}) to ({MAX_LON}, {MAX_LAT})')
print(f'  Density radius    : {DENSITY_RADIUS_DEG} deg (~1 km)')

## 3. Load and inspect competitor data

In [ ]:
comp = gpd.read_file(COMPETITOR_GEOJSON)

print(f'Raw competitor data loaded')
print(f'  Total features    : {len(comp):,}')
print(f'  Geometry types    : {comp.geometry.geom_type.value_counts().to_dict()}')
print(f'  CRS               : {comp.crs}')
print(f'  Coordinate range  : lon {comp.geometry.x.min():.4f} to {comp.geometry.x.max():.4f}')
print(f'                      lat {comp.geometry.y.min():.4f} to {comp.geometry.y.max():.4f}')
print()

# Key columns summary
print('Amenity breakdown:')
print(comp['amenity'].value_counts().to_string())
print()
print('Top 10 cuisine types:')
print(comp['cuisine'].value_counts().head(10).to_string())
print()
print('Delivery field values:')
print(comp['delivery'].value_counts().to_string() if 'delivery' in comp.columns else 'No delivery column')

## 4. Clean and clip to Delhi bbox

Keep only restaurants within the exact Delhi bounding box.
165 points outside the bbox are from the wider Overpass query
area — we drop them so they don't contaminate cell-level counts.

In [ ]:
# Ensure WGS84
if comp.crs is None:
    comp = comp.set_crs('EPSG:4326')
elif str(comp.crs) != 'EPSG:4326':
    comp = comp.to_crs('EPSG:4326')

# Keep only Point geometries (already all Points per inspection)
comp = comp[comp.geometry.geom_type == 'Point'].copy()

# Clip to Delhi bbox
before = len(comp)
comp = comp[
    (comp.geometry.x >= MIN_LON) & (comp.geometry.x <= MAX_LON) &
    (comp.geometry.y >= MIN_LAT) & (comp.geometry.y <= MAX_LAT)
].copy().reset_index(drop=True)
after = len(comp)
print(f'Clipped to Delhi bbox: {before} → {after} restaurants  (dropped {before-after} outside bbox)')

# Extract lon/lat as plain columns for fast numpy operations later
comp['lon'] = comp.geometry.x
comp['lat'] = comp.geometry.y

# ── Derive helper columns ──────────────────────────────────────────────

# Is this a known chain brand?
KNOWN_CHAINS = [
    'mcdonalds', 'kfc', 'dominos', 'pizza hut', 'subway', 'burger king',
    'haldirams', "mcdonald's", 'barbeque nation', 'wow momo', 'biryani by kilo',
    'behrouz biryani', 'fassos', 'box8', 'freshmenu', 'nirula', 'wimpy'
]
comp['is_chain'] = comp['name'].str.lower().fillna('').apply(
    lambda n: int(any(c in n for c in KNOWN_CHAINS))
)

# Does it offer delivery?
comp['has_delivery'] = comp['delivery'].fillna('unknown').apply(
    lambda v: 1 if str(v).lower() in ['yes', 'mo-su', 'only'] else 0
)

# Has a cuisine tag?
comp['has_cuisine'] = comp['cuisine'].notna().astype(int)

print(f'\nHelper columns computed:')
print(f'  Known chains      : {comp.is_chain.sum():,}')
print(f'  Has delivery tag  : {comp.has_delivery.sum():,}')
print(f'  Has cuisine tag   : {comp.has_cuisine.sum():,}')
print(f'  Has name          : {comp.name.notna().sum():,}')
comp[['name','amenity','cuisine','is_chain','has_delivery']].head(5)

## 5. Load the 500m grid

In [ ]:
# Load from 02a output if available, otherwise fall back to base grid
if os.path.exists(GRID_INPUT):
    gdf = pd.read_pickle(GRID_INPUT)
    print(f'Loaded grid from 02a output: {GRID_INPUT}')
else:
    gdf = gpd.read_file(GRID_FALLBACK)
    print(f'Loaded base grid: {GRID_FALLBACK}  (run 02a first to include population)')

if str(gdf.crs) != 'EPSG:4326':
    gdf = gdf.to_crs('EPSG:4326')

# Extract centroid coordinates for fast distance operations
if 'centroid_lon' not in gdf.columns:
    gdf['centroid_lon'] = gdf.geometry.centroid.x
    gdf['centroid_lat'] = gdf.geometry.centroid.y

print(f'Grid loaded  : {len(gdf):,} cells')
print(f'CRS          : {gdf.crs}')
gdf[['grid_id','centroid_lon','centroid_lat']].head(3)

## 6. Spatial join — count competitors inside each 500m cell

This is the core operation. geopandas `sjoin` checks which
restaurant points fall inside each 500m polygon.

In [ ]:
print('Running spatial join (restaurants inside 500m cells)...')

# Spatial join: each restaurant gets the grid_id of the cell it falls in
joined = gpd.sjoin(
    comp[['geometry','is_chain','has_delivery','has_cuisine','cuisine']],
    gdf[['grid_id','geometry']],
    how='left',
    predicate='within'
)

# Drop restaurants that fell outside any grid cell (edge cases)
in_grid = joined.dropna(subset=['grid_id'])
print(f'Restaurants matched to a grid cell : {len(in_grid):,} / {len(comp):,}')
print(f'Restaurants outside all cells      : {len(joined) - len(in_grid):,}  (edge/boundary cases)')

# ── Aggregate per cell ─────────────────────────────────────────────────

# Total competitor count
count_per_cell = in_grid.groupby('grid_id').size().rename('competitor_count')

# Chain count
chain_per_cell = in_grid.groupby('grid_id')['is_chain'].sum().rename('chain_count')

# Delivery-enabled count
delivery_per_cell = in_grid.groupby('grid_id')['has_delivery'].sum().rename('delivery_competitor_count')

# Cuisine diversity: number of unique cuisine types in the cell
cuisine_div = in_grid[in_grid['has_cuisine']==1].groupby('grid_id')['cuisine'].nunique().rename('cuisine_diversity')

# Merge all aggregations into one DataFrame
cell_stats = pd.DataFrame(index=gdf['grid_id'])
cell_stats = cell_stats.join(count_per_cell)
cell_stats = cell_stats.join(chain_per_cell)
cell_stats = cell_stats.join(delivery_per_cell)
cell_stats = cell_stats.join(cuisine_div)
cell_stats = cell_stats.fillna(0).astype(int)
cell_stats = cell_stats.reset_index()

print(f'\nAggregation complete.')
print(f'  Cells with at least 1 competitor : {(cell_stats.competitor_count > 0).sum():,}')
print(f'  Cells with zero competitors      : {(cell_stats.competitor_count == 0).sum():,}')
print(f'  Max competitors in one cell      : {cell_stats.competitor_count.max()}')
print(f'  Mean competitors per cell        : {cell_stats.competitor_count.mean():.2f}')
cell_stats.describe()

## 7. Nearest competitor distance using cKDTree

For each cell centroid, find the distance to the nearest restaurant.
cKDTree is much faster than a brute-force loop — handles 1000+ cells
and 1000+ restaurants in under 1 second.

Distance is in degrees then converted to approximate km.

In [ ]:
print('Computing nearest competitor distance using cKDTree...')

# Build KD-tree from competitor coordinates
comp_coords   = np.column_stack([comp['lon'].values, comp['lat'].values])
cell_centroids = np.column_stack([
    gdf['centroid_lon'].values,
    gdf['centroid_lat'].values
])

tree = cKDTree(comp_coords)

# Query nearest neighbour for every cell centroid
# k=1 → nearest single competitor
# k=3 → we also get 2nd and 3rd nearest for averaging
distances_k1, _ = tree.query(cell_centroids, k=1)
distances_k3, _ = tree.query(cell_centroids, k=min(3, len(comp)))

# Convert from degrees to approximate km
# At Delhi latitude (~28.7°), 1 degree ≈ 88 km longitude, ≈ 111 km latitude
# Using 1 degree ≈ 100 km as a simple approximation
DEG_TO_KM = 100.0

nearest_km = distances_k1 * DEG_TO_KM

if distances_k3.ndim == 1:
    avg_3_km = distances_k3 * DEG_TO_KM
else:
    avg_3_km = distances_k3.mean(axis=1) * DEG_TO_KM

# Attach to gdf indexed by position
gdf['nearest_competitor_km']  = nearest_km
gdf['avg_3_nearest_comp_km']  = avg_3_km

print(f'Nearest competitor distance computed.')
print(f'  Min distance (most saturated cell) : {nearest_km.min():.3f} km')
print(f'  Max distance (most isolated cell)  : {nearest_km.max():.3f} km')
print(f'  Mean distance                      : {nearest_km.mean():.3f} km')
print(f'  Median distance                    : {np.median(nearest_km):.3f} km')

## 8. Competitor density in 1km radius

Count all competitors within approximately 1km of each cell centroid.
This captures the competitive pressure beyond the cell boundary —
a low-count cell surrounded by dense competitors is still saturated.

In [ ]:
print(f'Computing competitor density within {DENSITY_RADIUS_DEG} deg (~1 km) radius...')

# cKDTree query_ball_point returns indices of all competitors within radius
nearby_indices = tree.query_ball_point(cell_centroids, r=DENSITY_RADIUS_DEG)

# Count all competitors within radius
gdf['competitor_density_1km'] = [len(idx) for idx in nearby_indices]

# Count delivery-enabled competitors within radius
delivery_flags = comp['has_delivery'].values
gdf['delivery_comp_density_1km'] = [
    int(delivery_flags[idx].sum()) for idx in nearby_indices
]

# Count chain competitors within radius
chain_flags = comp['is_chain'].values
gdf['chain_density_1km'] = [
    int(chain_flags[idx].sum()) for idx in nearby_indices
]

print(f'1km density computed.')
print(f'  Mean competitors in 1km radius  : {gdf.competitor_density_1km.mean():.2f}')
print(f'  Max competitors in 1km radius   : {gdf.competitor_density_1km.max()}')
print(f'  Cells with 0 competitors in 1km : {(gdf.competitor_density_1km == 0).sum():,}')

## 9. Merge all competitor features onto the grid

In [ ]:
# Merge the aggregated cell stats (from spatial join) into gdf
gdf = gdf.merge(cell_stats, on='grid_id', how='left')

# Fill any NaN from cells that had zero competitors
COMP_JOIN_COLS = [
    'competitor_count', 'chain_count',
    'delivery_competitor_count', 'cuisine_diversity'
]
gdf[COMP_JOIN_COLS] = gdf[COMP_JOIN_COLS].fillna(0).astype(int)

# ── Derived features ───────────────────────────────────────────────────

# Market saturation score: how competitive is this cell?
# Normalised to 0-1 range using 95th percentile as ceiling
p95_count   = gdf['competitor_density_1km'].quantile(0.95)
gdf['market_saturation'] = (
    gdf['competitor_density_1km'] / max(p95_count, 1)
).clip(0, 1)

# Chain ratio: proportion of nearby competitors that are known chains
gdf['chain_ratio'] = (
    gdf['chain_density_1km'] / (gdf['competitor_density_1km'] + 1e-9)
).clip(0, 1)

# Opportunity gap: cells with LOW competitor density relative to population
# are underserved markets — high pop + low competition = opportunity
# Only meaningful if pop_density column exists from 02a
if 'pop_density' in gdf.columns:
    p95_pop   = gdf['pop_density'].quantile(0.95)
    norm_pop  = (gdf['pop_density'] / max(p95_pop, 1)).clip(0, 1)
    norm_comp = gdf['market_saturation']
    gdf['opportunity_gap'] = (norm_pop - norm_comp).clip(0, 1)
    print('opportunity_gap computed using real WorldPop population density.')
else:
    gdf['opportunity_gap'] = (1 - gdf['market_saturation']).clip(0, 1)
    print('opportunity_gap computed without population (run 02a first for better signal).')

# Delivery gap: high foot traffic area but low delivery competitors nearby
gdf['delivery_gap'] = (
    1 - (gdf['delivery_comp_density_1km'] / (gdf['competitor_density_1km'] + 1e-9))
).clip(0, 1)

# All competitor feature columns
COMP_FEATURE_COLS = [
    'competitor_count',          # restaurants inside this 500m cell
    'chain_count',               # chains inside this cell
    'delivery_competitor_count', # delivery-enabled inside this cell
    'cuisine_diversity',         # unique cuisine types inside this cell
    'nearest_competitor_km',     # distance to nearest restaurant (km)
    'avg_3_nearest_comp_km',     # average of 3 nearest (less noisy)
    'competitor_density_1km',    # all restaurants within 1km radius
    'delivery_comp_density_1km', # delivery restaurants within 1km
    'chain_density_1km',         # chains within 1km
    'market_saturation',         # normalised competition score 0-1
    'chain_ratio',               # fraction of competitors that are chains
    'opportunity_gap',           # high pop low competition signal 0-1
    'delivery_gap',              # delivery underserved signal 0-1
]

print(f'\nAll competitor features computed:')
for col in COMP_FEATURE_COLS:
    nz = (gdf[col] > 0).sum()
    print(f'  {col:<30}  non-zero: {nz:,}  mean: {gdf[col].mean():.3f}  max: {gdf[col].max():.3f}')

## 10. NaN and Inf check

In [ ]:
print('Quality check:')
all_clean = True
for col in COMP_FEATURE_COLS:
    nans = gdf[col].isna().sum()
    infs = np.isinf(gdf[col].values).sum()
    status = 'OK'
    if nans > 0:
        status = f'WARNING {nans} NaNs — auto-filling with 0'
        gdf[col] = gdf[col].fillna(0)
        all_clean = False
    if infs > 0:
        status += f'  WARNING {infs} Infs — auto-fixing'
        gdf[col] = gdf[col].replace([np.inf, -np.inf], 0)
        all_clean = False
    print(f'  {col:<30} {status}')

if all_clean:
    print('\nAll competitor feature columns are clean.')
else:
    print('\nIssues auto-fixed.')

## 11. Visualise competitor distribution on the grid

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

plot_specs = [
    ('competitor_count',       'Competitors inside cell',           'OrRd'),
    ('competitor_density_1km', 'Competitor density (1km radius)',   'YlOrRd'),
    ('nearest_competitor_km',  'Distance to nearest competitor (km)','RdYlGn'),
    ('market_saturation',      'Market saturation score (0-1)',     'Reds'),
    ('opportunity_gap',        'Opportunity gap score (0-1)',       'Greens'),
    ('delivery_gap',           'Delivery gap score (0-1)',          'Blues'),
]

for i, (col, title, cmap) in enumerate(plot_specs):
    ax = axes[i]
    active = gdf[gdf[col] > 0]
    zero   = gdf[gdf[col] == 0]

    if len(active) > 0:
        active.plot(
            column=col, ax=ax, cmap=cmap,
            scheme='quantiles', k=6,
            legend=True, edgecolor='none', alpha=0.85,
            aspect=None
        )
    if len(zero) > 0:
        zero.plot(ax=ax, color='#F0F0F0', edgecolor='none', aspect=None)

    # Overlay restaurant points
    ax.scatter(
        comp['lon'], comp['lat'],
        s=3, c='black', alpha=0.4, zorder=5
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude') ; ax.set_ylabel('Latitude')

plt.suptitle(f'Competitor features — Delhi 500m grid  (n={len(comp):,} restaurants)',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('data/processed/competitor_features_maps.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved to data/processed/competitor_features_maps.png')

## 12. Cuisine type distribution — what is the competition landscape?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top cuisines
top_cuisines = comp['cuisine'].value_counts().head(15)
top_cuisines.plot(
    kind='barh', ax=axes[0],
    color='#D85A30', edgecolor='white'
)
axes[0].set_title('Top 15 cuisine types in Delhi (OSM data)')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Chain vs independent
chain_counts = comp['is_chain'].value_counts()
labels = ['Independent', 'Known chain']
colors = ['#1D9E75', '#534AB7']
axes[1].pie(
    chain_counts.values, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Chain vs independent restaurants')

plt.tight_layout()
plt.savefig('data/processed/competitor_cuisine_analysis.png', dpi=110, bbox_inches='tight')
plt.show()

print(f'Total restaurants in Delhi bbox : {len(comp):,}')
print(f'Named restaurants               : {comp.name.notna().sum():,}')
print(f'With cuisine tag                : {comp.has_cuisine.sum():,}')
print(f'With delivery tag = yes         : {comp.has_delivery.sum():,}')
print(f'Known chain brands              : {comp.is_chain.sum():,}')

## 13. Correlation with profitability label (if available)

In [ ]:
if 'profitable' in gdf.columns:
    print('Correlation of competitor features with profitability label:')
    corr = gdf[COMP_FEATURE_COLS + ['profitable']].corr()['profitable']
    corr = corr.drop('profitable').sort_values()

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#D85A30' if v < 0 else '#1D9E75' for v in corr.values]
    ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title('Competitor feature correlation with profitability')
    ax.set_xlabel('Pearson correlation')
    plt.tight_layout()
    plt.show()
    print(corr.to_string())
else:
    print('No profitability label yet — run 02_features.ipynb to add labels.')
    print('Skipping correlation check.')

## 14. Save output

In [ ]:
gdf.to_pickle(OUT_PKL)

print(f'Saved: {OUT_PKL}')
print(f'  Shape            : {gdf.shape}')
print(f'  Competitor cols  : {COMP_FEATURE_COLS}')
print(f'  NaN total        : {gdf[COMP_FEATURE_COLS].isna().sum().sum()}')

## 15. Final summary

In [ ]:
print('=' * 60)
print('  COMPETITOR FEATURE EXTRACTION COMPLETE')
print('=' * 60)
print(f'  Restaurants loaded         : 1,227')
print(f'  Within Delhi bbox          : {len(comp):,}')
print(f'  Grid cells                 : {len(gdf):,}')
print(f'  Cells with competitors     : {(gdf.competitor_count > 0).sum():,}')
print(f'  Cells with zero (cold)     : {(gdf.competitor_count == 0).sum():,}')
print()
print(f'  Mean competitors per cell  : {gdf.competitor_count.mean():.2f}')
print(f'  Max competitors in 1 cell  : {gdf.competitor_count.max()}')
print(f'  Mean 1km density           : {gdf.competitor_density_1km.mean():.2f}')
print(f'  Mean distance to nearest   : {gdf.nearest_competitor_km.mean():.3f} km')
print()
print(f'  Output saved to: {OUT_PKL}')
print()
print('  Next step: open 02_features.ipynb')
print('  Load with:')
print('    gdf = pd.read_pickle("data/processed/grid_with_competitors.pkl")')
print('  competitor_count is now REAL OSM data — not synthetic.')
print('=' * 60)

## 16. What to change in 02_features.ipynb

**Change 1 — Load from this notebook's output:**

```python
# REPLACE:
gdf = gpd.read_file('data/processed/grid.geojson')

# WITH (if you ran both 02a and 02b):
gdf = pd.read_pickle('data/processed/grid_with_competitors.pkl')
# This already has population columns from 02a AND competitor columns from 02b
```

**Change 2 — In generate_synthetic_data(), remove these lines:**

```python
# REMOVE (now using real data):
gdf['competitor_count'] = np.random.poisson(3, len(gdf))

# KEEP these (still synthetic — no real data yet):
gdf['income_index']           = np.random.uniform(0.3, 1.0, len(gdf))
gdf['road_density']           = np.random.uniform(0.2, 1.0, len(gdf))
gdf['transit_stops']          = np.random.poisson(2, len(gdf))
gdf['warehouse_proximity_km'] = np.random.uniform(0.5, 15.0, len(gdf))
```

**Change 3 — Add extra competitor features to your feature list:**

```python
# In build_feature_matrix(), include these from the competitor notebook:
EXTRA_COMP_FEATURES = [
    'competitor_density_1km',  # better signal than raw count
    'nearest_competitor_km',   # isolation proxy
    'market_saturation',       # normalised competition 0-1
    'opportunity_gap',         # underserved area signal
    'delivery_gap',            # delivery underserved signal
    'chain_ratio',             # market maturity indicator
    'cuisine_diversity',       # variety of competition
]
# These are the most useful for LightGBM — better than raw competitor_count alone
```

**Your real data coverage after running 02a + 02b:**

| Feature | Source | Status |
|---|---|---|
| `pop_density` | WorldPop TIF | REAL |
| `pop_density_log` | WorldPop TIF | REAL |
| `competitor_count` | OSM Overpass | REAL |
| `competitor_density_1km` | OSM Overpass | REAL |
| `nearest_competitor_km` | OSM Overpass | REAL |
| `market_saturation` | OSM Overpass | REAL |
| `opportunity_gap` | WorldPop + OSM | REAL |
| `income_index` | — | synthetic |
| `road_density` | — | synthetic (use OSMnx for real) |
| `transit_stops` | — | synthetic (use OSM for real) |